# M1 — Data Exploration

Load price and macro data from `data/state.db`.

**Prerequisites:** Run both ingestion scripts before opening this notebook:
```bash
uv run python scripts/ingest_prices.py
uv run python scripts/ingest_macro.py
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

from config import load_config

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')
universe = load_config('universe')
etf_tickers = universe.ticker_list  # 10 ETFs, excludes SPY benchmark
print('Universe:', etf_tickers)

In [ ]:
df = pd.read_sql(
    text('SELECT date, ticker, close, adj_close FROM prices ORDER BY date'),
    engine,
    parse_dates=['date'],
)
etf_df = df[df['ticker'].isin(etf_tickers)]
print(f'{len(etf_df):,} rows | {etf_df["date"].min().date()} → {etf_df["date"].max().date()}')
etf_df.head()

In [ ]:
# Normalise to 100 at the start of the series for comparability
pivot = etf_df.pivot(index='date', columns='ticker', values='adj_close')
normalised = pivot / pivot.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
for ticker in etf_tickers:
    if ticker in normalised.columns:
        normalised[ticker].plot(ax=ax, label=ticker, linewidth=1.2)

ax.set_title('Sector ETF Adjusted Close — Normalised to 100', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Normalised price (base = 100)')
ax.legend(ncol=2, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Basic summary stats
returns = pivot.pct_change().dropna()
summary = pd.DataFrame({
    'Ann. Return': returns.mean() * 252,
    'Ann. Volatility': returns.std() * (252 ** 0.5),
    'Sharpe (rf=0)': (returns.mean() / returns.std()) * (252 ** 0.5),
    'Max Drawdown': (pivot / pivot.cummax() - 1).min(),
}).round(3)
summary.sort_values('Sharpe (rf=0)', ascending=False)

## Macro Data — Yield Curve Spread & VIX

T10Y2Y (10Y-2Y spread) and VIXCLS (VIX) are the two primary macro regime signals.
Negative spread = inverted yield curve = recession signal.

In [ ]:
macro_df = pd.read_sql(
    text("SELECT date, series_id, value FROM macro WHERE series_id IN ('T10Y2Y', 'VIXCLS') ORDER BY date"),
    engine,
    parse_dates=['date'],
)
macro_pivot = macro_df.pivot(index='date', columns='series_id', values='value')
print(f'Macro rows: {len(macro_df):,} | {macro_df["date"].min().date()} → {macro_df["date"].max().date()}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

macro_pivot['T10Y2Y'].plot(ax=ax1, color='steelblue', linewidth=1.2)
ax1.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax1.set_title('10Y-2Y Treasury Spread (T10Y2Y)', fontsize=12)
ax1.set_ylabel('Spread (%)')
ax1.grid(alpha=0.3)

macro_pivot['VIXCLS'].plot(ax=ax2, color='darkorange', linewidth=1.2)
ax2.axhline(20, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='VIX=20 (elevated)')
ax2.set_title('VIX (VIXCLS)', fontsize=12)
ax2.set_ylabel('VIX level')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## News Coverage — Article Count Heatmap

Article counts per sector ETF per calendar month from `news_raw`.

**Prerequisites:** Run `scripts/ingest_news.py` first.

In [ ]:
import numpy as np

news_df = pd.read_sql(
    text("SELECT sector, timestamp FROM news_raw"),
    engine,
    parse_dates=["timestamp"],
)

if news_df.empty:
    print("No news data yet — run scripts/ingest_news.py first.")
else:
    news_df["month"] = news_df["timestamp"].dt.to_period("M").astype(str)
    counts = news_df.groupby(["sector", "month"]).size().unstack(fill_value=0)
    counts = counts.sort_index()

    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(counts.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(counts.columns)))
    ax.set_xticklabels(counts.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(counts.index)))
    ax.set_yticklabels(counts.index)
    ax.set_title("News Articles per Sector per Month", fontsize=13)
    plt.colorbar(im, ax=ax, label="Article count")
    for i in range(len(counts.index)):
        for j in range(len(counts.columns)):
            v = counts.values[i, j]
            ax.text(j, i, str(v), ha="center", va="center", fontsize=7,
                    color="white" if v > counts.values.max() * 0.6 else "black")
    plt.tight_layout()
    plt.show()
    print(f"Total articles: {news_df.shape[0]:,} | Sectors: {news_df['sector'].nunique()} | "
          f"Date range: {news_df['timestamp'].min().date()} → {news_df['timestamp'].max().date()}")

## Polymarket — Curated Market Probabilities

Current implied probabilities for macro-relevant prediction markets.
Sector-impact mappings are defined in `config/polymarket_markets.yaml`.

**Prerequisites:** Run `scripts/ingest_polymarket.py` first.

In [ ]:
from ingestion.polymarket import load_curated_markets

pm_df = pd.read_sql(
    text("""
        SELECT market_id, timestamp, question, implied_prob, volume, end_date
        FROM polymarket_raw
        ORDER BY market_id, timestamp
    """),
    engine,
    parse_dates=["timestamp", "end_date"],
)

if pm_df.empty:
    print("No Polymarket data yet — run scripts/ingest_polymarket.py first.")
else:
    # Latest snapshot per market
    latest = (
        pm_df.sort_values("timestamp")
        .groupby("market_id")
        .last()
        .reset_index()
        .sort_values("implied_prob", ascending=False)
    )

    # ── Bar chart: current implied probabilities ──
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ["#2ecc71" if p >= 0.5 else "#e74c3c" for p in latest["implied_prob"]]
    bars = ax.barh(range(len(latest)), latest["implied_prob"], color=colors, alpha=0.8)
    ax.set_yticks(range(len(latest)))
    ax.set_yticklabels(
        [q[:72] + "…" if len(q) > 72 else q for q in latest["question"]],
        fontsize=8,
    )
    ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Implied probability (YES)")
    ax.set_title("Polymarket — Current Macro Market Probabilities", fontsize=13)
    for i, bar in enumerate(bars):
        p = latest["implied_prob"].iloc[i]
        ax.text(
            p + 0.01 if p < 0.9 else p - 0.06,
            bar.get_y() + bar.get_height() / 2,
            f"{p:.0%}",
            va="center",
            fontsize=8,
        )
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    display_cols = ["market_id", "implied_prob", "volume", "end_date"]
    print(latest[display_cols].to_string(index=False))

    # ── Price history (for markets with multiple snapshots) ──
    multi = pm_df.groupby("market_id").filter(lambda g: len(g) > 1)
    if not multi.empty:
        fig, ax = plt.subplots(figsize=(14, 5))
        for mid, grp in multi.groupby("market_id"):
            ax.plot(grp["timestamp"], grp["implied_prob"], marker=".", label=mid, linewidth=1.2)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
        ax.set_title("Polymarket — Probability History (markets with multiple snapshots)", fontsize=12)
        ax.set_ylabel("Implied probability (YES)")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()